# Stroke Diagnosis - Comprehensive ML Pipeline
This notebook demonstrates a robust, production-ready process of Exploratory Data Analysis (EDA), Data Preprocessing, and Model Training for Stroke Prediction.

### Key Improvements:
- Proper **Train/Test Split BEFORE SMOTE** to prevent data leakage.
- Use of `imblearn.pipeline.Pipeline` for rigorous Cross Validation.
- Detailed EDA with visualizations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import pickle

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")


## 1. Data Loading & Basic Inspection


In [ ]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
display(df.head())
print(f"Dataset shape: {df.shape}")


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())


## 2. Exploratory Data Analysis (EDA)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['age'], bins=30, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Age Distribution')

sns.histplot(df['avg_glucose_level'], bins=30, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Average Glucose Level Distribution')

sns.histplot(df['bmi'].dropna(), bins=30, kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('BMI Distribution')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='stroke', palette='Set2')
plt.title('Stroke Class Distribution (Highly Imbalanced)')
plt.show()

stroke_counts = df['stroke'].value_counts()
print(f"Stroke 0: {stroke_counts[0]} ({stroke_counts[0]/len(df)*100:.2f}%)")
print(f"Stroke 1: {stroke_counts[1]} ({stroke_counts[1]/len(df)*100:.2f}%)")


## 3. Data Preprocessing & Feature Engineering


In [ ]:
# Drop unnecessary ID column
df = df.drop(columns=['id'])

# Remove 'Other' gender as it's statistically insignificant (only 1 instance)
df = df[df['gender'] != 'Other']

# Impute missing BMI with median (or mean, since it's slightly right-skewed but median is safer)
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Encoding Binary variables
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)
df['gender']       = (df['gender'] == 'Female').astype(int)

# One-hot encoding for Multi-class Categorical variables
df = pd.get_dummies(df, columns=['work_type', 'Residence_type', 'smoking_status'], drop_first=False)

print("Features after encoding:")
print(df.columns.tolist())


## 4. Train-Test Split (CRITICAL STEP)
To avoid Data Leakage, we must split the data **before** applying any oversampling techniques like SMOTE.


In [ ]:
X = df.drop('stroke', axis=1)
y = df['stroke']

# Save metadata for backend
feature_cols = X.columns.tolist()
with open('backend/stroke_metadata.pkl', 'wb') as f:
    pickle.dump({'feature_cols': feature_cols}, f)

# Stratify=y ensures the 5% stroke ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")


## 5. Model Training Pipeline with SMOTE
We use `imblearn.pipeline.Pipeline` which correctly applies SMOTE only during the `fit` phase, ensuring validation sets inside cross-validation are NOT oversampled.


In [ ]:
# Define models with Imblearn Pipelines
pipelines = {
    "Logistic Regression": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
    ]),
    "Gradient Boosting": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', GradientBoostingClassifier(random_state=42))
    ]),
    "Hist Gradient Boosting": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', HistGradientBoostingClassifier(random_state=42))
    ]),
    "SVM": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    "KNN": ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ])
}


In [ ]:
trained_models = {}
metrics = []

print("Training & Evaluating Models...")
for name, pipe in pipelines.items():
    # Fit on training data
    pipe.fit(X_train, y_train)
    
    # Predict on test data (SMOTE is NOT applied to test data by the pipeline)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    metrics.append({
        "Model": name,
        "AUC-ROC": auc,
        "F1-Score": f1,
        "Accuracy": acc
    })
    
    trained_models[name] = pipe
    print(f"✅ {name} evaluated.")

# Voting Ensemble
print("\nBuilding Voting Ensemble...")
# VotingClassifier requires estimators that support predict_proba if voting='soft'
estimators = [(n, m) for n, m in trained_models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting='soft')

# Note: Since the base estimators are already pipelines with SMOTE, 
# VotingClassifier will run them in parallel. 
# A cleaner way is to do SMOTE -> Voting(Clf1, Clf2), but since our pipelines already 
# encompass everything nicely and we fit them all, we can just fit the VotingClassifier.

ensemble_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('clf', VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000, random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('hgb', HistGradientBoostingClassifier(random_state=42))
        ],
        voting='soft'
    ))
])

ensemble_pipe.fit(X_train, y_train)
y_pred_ens = ensemble_pipe.predict(X_test)
y_proba_ens = ensemble_pipe.predict_proba(X_test)[:, 1]

metrics.append({
    "Model": "Voting Ensemble",
    "AUC-ROC": roc_auc_score(y_test, y_proba_ens),
    "F1-Score": f1_score(y_test, y_pred_ens),
    "Accuracy": accuracy_score(y_test, y_pred_ens)
})
trained_models["Voting Ensemble"] = ensemble_pipe
print("✅ Voting Ensemble evaluated.")


## 6. Model Evaluation Summary


In [ ]:
metrics_df = pd.DataFrame(metrics).sort_values(by="AUC-ROC", ascending=False)
display(metrics_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x='AUC-ROC', y='Model', palette='viridis')
plt.title('Model Comparison by AUC-ROC')
plt.xlim(0.6, 1.0)
plt.show()


## 7. Export Models for Backend


In [ ]:
with open('backend/stroke_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)

print("Exported robust models to backend/stroke_models.pkl successfully!")
